In [1]:
import numpy as np
import pandas as pd
from IPython.core.inputtransformer2 import cell_magic
from pyparsing import countedArray

got train_transaction and train_identity datasets. transaction has all the core transaction info + a bunch of derived columns, identity has device/network/browser stuff for some transactions

In [2]:
train_trans = pd.read_csv('../data/raw/train_transaction.csv')
train_id = pd.read_csv('../data/raw/train_identity.csv')


both datasets together help the model find relationships between transaction info and identity info to predict fraud. understanding the structure first (columns, what they mean) matters before jumping into anything

In [3]:
train = train_trans.merge(train_id, how='left', on='TransactionID')
print(train.shape)
print(f'{train.memory_usage(index=True, deep=True).sum():,}')

(590540, 434)
2,636,093,316


analyzing target variable is essential:
**how many unique values it contains**
**among those unique values, what is the percentage of each value**
my observation is that almost **96.5%** of user are non-fraud/0 and **~3.5%** of users are fraudster/1 which suggest higly imbalanced dataset.

In [4]:
target = train_trans['isFraud']
print(target.value_counts(normalize=True))


isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [5]:
train_trans_len = len(train_trans)
print(train_trans_len)

590540


ran this on train_trans before any downcasting. just checking null count + null % per column to see which ones are mostly empty, which are clean - helps figure out which V columns might be related later

In [6]:
null_train_trans = pd.DataFrame({
    'null_count': train_trans.isnull().sum(),
    'null_pct': ((train_trans.isnull().sum()/train_trans_len) * 100).round(2)
}).sort_values('null_pct', ascending=False)
pd.set_option('display.max_rows', None)
print(null_train_trans[null_train_trans['null_count'] > 0])

               null_count  null_pct
dist2              552913     93.63
D7                 551623     93.41
D13                528588     89.51
D14                528353     89.47
D12                525823     89.04
D6                 517353     87.61
D9                 515614     87.31
D8                 515614     87.31
V153               508595     86.12
V140               508595     86.12
V150               508589     86.12
V149               508595     86.12
V155               508595     86.12
V147               508595     86.12
V146               508595     86.12
V145               508589     86.12
V144               508589     86.12
V143               508589     86.12
V142               508595     86.12
V141               508595     86.12
V152               508589     86.12
V157               508595     86.12
V139               508595     86.12
V138               508595     86.12
V154               508595     86.12
V151               508589     86.12
V165               508589   

this just pulls out columns with zero nulls, since those are the ones safe to downcast straight to int without worrying about NaN

In [7]:
zero_null_cols = train_trans.columns[train_trans.isnull().sum() == 0].tolist()
print(zero_null_cols)
print(train_trans[zero_null_cols].dtypes.value_counts())

['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']
float64    15
int64       4
str         1
Name: count, dtype: int64


max/min just tells me the biggest/smallest value in the column. picking the dtype (uint8 vs int16 etc) is a separate step - I check if that range fits inside the dtype's limit, then use the smallest one that fits

In [8]:
int_cols = train_trans[zero_null_cols].select_dtypes(include=['int64']).columns.tolist()
print(train_trans[int_cols].agg(['min', 'max']))

     TransactionID  isFraud  TransactionDT  card1
min        2987000        0          86400   1000
max        3577539        1       15811131  18396


for float we don't really pick a size per column like we did for int. float64 -> float32 is basically the only real option, since float32's range is already way bigger than anything in this dataset. the actual risk is losing precision, not overflowing, checked that separately below

In [9]:
float_cols = train_trans[zero_null_cols].select_dtypes(include=['float64']).columns.tolist()
print(train_trans[float_cols].agg(['min', 'max']))

     TransactionAmt      C1      C2    C3      C4     C5      C6      C7  \
min           0.251     0.0     0.0   0.0     0.0    0.0     0.0     0.0   
max       31937.391  4685.0  5691.0  26.0  2253.0  349.0  2253.0  2255.0   

         C8     C9     C10     C11     C12     C13     C14  
min     0.0    0.0     0.0     0.0     0.0     0.0     0.0  
max  3331.0  210.0  3257.0  3188.0  3188.0  2918.0  1429.0  


checked if TransactionAmt is all whole numbers, got False - so it actually has decimals, can't downcast this to int, has to stay float

In [10]:
print((train_trans['TransactionAmt'] % 1 == 0).all())

False


converted each int column to whatever dtype actually fits its min/max range, instead of leaving everything as int64

In [11]:
train_trans['isFraud'] = train_trans['isFraud'].astype(np.uint8)
train_trans['card1'] = train_trans['card1'].astype(np.uint16)
train_trans['TransactionDT'] = train_trans['TransactionDT'].astype(np.uint32)
train_trans['TransactionID'] = train_trans['TransactionID'].astype(np.uint32)

In [12]:
#transaction amount is float, should float downcasted
train_trans['TransactionAmt'] = train_trans['TransactionAmt'].astype(np.float32)

In [13]:
train_trans['C1'] = train_trans['C1'].astype(np.uint16)
train_trans['C2'] = train_trans['C2'].astype(np.uint16)
train_trans['C3'] = train_trans['C3'].astype(np.uint8)
train_trans['C4'] = train_trans['C4'].astype(np.uint16)
train_trans['C5'] = train_trans['C5'].astype(np.uint16)
train_trans['C6'] = train_trans['C6'].astype(np.uint16)
train_trans['C7'] = train_trans['C7'].astype(np.uint16)
train_trans['C8'] = train_trans['C8'].astype(np.uint16)
train_trans['C9'] = train_trans['C9'].astype(np.uint8)
train_trans['C10'] = train_trans['C10'].astype(np.uint16)
train_trans['C11'] = train_trans['C11'].astype(np.uint16)
train_trans['C12'] = train_trans['C12'].astype(np.uint16)
train_trans['C13'] = train_trans['C13'].astype(np.uint16)
train_trans['C14'] = train_trans['C14'].astype(np.uint16)

wasn't worried about float32 being "too small" for the values, range is fine. the real risk is precision - float32 only keeps ~7 sig figs so very close numbers can end up looking identical after conversion. tested it on TransactionAmt and got 0 loss so we're fine here

In [14]:
original = train_trans['TransactionAmt'].astype(np.float64)
converted = train_trans['TransactionAmt'].astype(np.float32)
diff = (original - converted).abs()
print(f"Max precision loss: {diff.max()}")
print(f"Mean precision loss: {diff.mean()}")

Max precision loss: 0.0
Mean precision loss: 0.0


In [15]:
float64_cols = train_trans.select_dtypes(include='float64').columns.tolist()
print(float64_cols)

['card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', 'V71', 'V72', 'V73', 'V74', 'V75', 'V76', 'V77', 'V78', 'V79', 'V80', 'V81', 'V82', 'V83', 'V84', 'V85', 'V86', 'V87', 'V88', 'V89', 'V90', 'V91', 'V92', 'V93', 'V94', 'V95', 'V96', 'V97', 'V98', 'V99', 'V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V107', 'V108', 'V109', 'V110', 'V111', 'V112', 'V113', 'V114', 'V115', 'V116', 'V117', 'V118', 'V11

In [16]:
train_trans[float64_cols] = train_trans[float64_cols].astype(np.float32)

In [17]:
after = train_trans.memory_usage(deep=True).sum() / 1e6
print(f"After: {after:.2f} MB")

After: 1243.95 MB


doing null check on train_id by itself, not after merge. if i merge first, every row that never had identity data gets NaN added on top, which inflates the null % and hides the real pattern that existed in train_id alone

In [18]:
null_train_id = pd.DataFrame({
    'null_count': train_id.isnull().sum(),
    'null_pct': ((train_id.isnull().sum()/len(train_id)) * 100).round(2)
}).sort_values('null_pct', ascending=False)
pd.set_option('display.max_rows', None)
print(null_train_id[null_train_id['null_count'] > 0])

            null_count  null_pct
id_24           139486     96.71
id_25           139101     96.44
id_07           139078     96.43
id_08           139078     96.43
id_21           139074     96.42
id_22           139064     96.42
id_23           139064     96.42
id_27           139064     96.42
id_26           139070     96.42
id_18            99120     68.72
id_04            77909     54.02
id_03            77909     54.02
id_33            70944     49.19
id_10            69307     48.05
id_09            69307     48.05
id_30            66668     46.22
id_32            66647     46.21
id_34            66428     46.06
id_14            64189     44.50
DeviceInfo       25567     17.73
id_13            16913     11.73
id_16            14893     10.33
id_06             7368      5.11
id_05             7368      5.11
id_20             4972      3.45
id_19             4915      3.41
id_17             4864      3.37
id_31             3951      2.74
DeviceType        3423      2.37
id_02     

In [19]:
no_null_cols1=  train_id.columns[train_id.isnull().sum() == 0].tolist()
print(no_null_cols1)
print(train_id[no_null_cols1].dtypes.value_counts())

['TransactionID', 'id_01', 'id_12']
int64      1
float64    1
str        1
Name: count, dtype: int64


In [20]:
int_cols1 = train_id[no_null_cols1].select_dtypes(include=['int64']).columns.tolist()
print(train_id[int_cols1].agg(['min', 'max']))

     TransactionID
min        2987004
max        3577534


In [21]:
id01 = train_id['id_01']
print(id01.value_counts(normalize=False))


id_01
-5.0      82170
 0.0      19555
-10.0     11257
-20.0     11211
-15.0      5674
-25.0      4623
-45.0      2143
-35.0      1622
-40.0      1385
-100.0     1012
-50.0       709
-30.0       682
-95.0       428
-60.0       410
-55.0       320
-80.0       220
-90.0       214
-70.0        97
-65.0        93
-85.0        87
-75.0        83
-18.0        23
-12.0        15
-11.0        15
-6.0         15
-16.0        13
-21.0        12
-7.0         10
-14.0        10
-19.0         9
-31.0         9
-17.0         9
-26.0         8
-27.0         6
-87.0         5
-23.0         5
-22.0         5
-37.0         5
-9.0          4
-62.0         4
-13.0         4
-44.0         3
-96.0         3
-38.0         3
-64.0         2
-99.0         2
-53.0         2
-88.0         2
-61.0         2
-46.0         2
-29.0         2
-8.0          2
-71.0         2
-56.0         2
-72.0         1
-58.0         1
-42.0         1
-76.0         1
-34.0         1
-54.0         1
-28.0         1
-94.0         1
-2

In [22]:
print(id01.min(), id01.max())

-100.0 0.0


id_01 has negatives (-100 to 0), so uint8 is out since it's unsigned. int8 covers -128 to 127 so that works fine

In [23]:
train_id['id_01'] = train_id['id_01'].astype(np.int8)
train_id['TransactionID'] = train_id['TransactionID'].astype(np.int32)

In [24]:
float64_cols1 = train_id.select_dtypes(include='float64').columns.tolist()
train_id[float64_cols1] = train_id[float64_cols1].astype(np.float32)

In [25]:
original1 = train_id['id_01'].astype(np.float64)
converted1 = train_id['id_01'].astype(np.float32)
diff = (original1 - converted1).abs()
print(f"Max precision loss: {diff.max()}")
print(f"Mean precision loss: {diff.mean()}")

Max precision loss: 0.0
Mean precision loss: 0.0


In [26]:
after_id = train_id.memory_usage(deep=True).sum() / 1e6
print(f"train identity after: {after_id:.2f} MB")

train identity after: 135.82 MB


In [27]:
train = train_trans.merge(train_id, how='left', on='TransactionID')
print(train.shape)
print(f'{train.memory_usage(index=True, deep=True).sum():,}')

(590540, 434)
1,665,836,096


object dtype was storing the same repeated strings over and over (like gmail.com thousands of times). category just stores the unique values once and uses small ints to point to them, that's where the memory drop came from

In [28]:
obj_cols = train.select_dtypes(include='object').columns.tolist()
print(obj_cols)
print(f"Object cols memory: {train[obj_cols].memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Non-object memory: {train.drop(columns=obj_cols).memory_usage(deep=True).sum() / 1e6:.1f} MB")

['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


C:\Users\moizs\AppData\Local\Temp\ipykernel_1816\18688683.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = train.select_dtypes(include='object').columns.tolist()


Object cols memory: 732.2 MB
Non-object memory: 933.6 MB


In [29]:
for col in obj_cols:
    train[col] = train[col].astype('category')

print(f"After category encoding: {train.memory_usage(deep=True).sum() / 1e6:.1f} MB")

After category encoding: 953.9 MB


a bunch of V columns share the exact same null % which isn't random - they were probably computed together from the same source data, so when that source was missing the whole batch goes null together

In [30]:
null_pct = null_train_trans['null_pct']

In [31]:
null_pct_cols = null_pct.groupby(null_pct).apply(lambda n: n.index.tolist())


In [32]:
for pct, cols in null_pct_cols.items():
    print(f"{pct}: {cols}")

0.0: ['V318', 'V317', 'V316', 'V305', 'card1', 'C1', 'C2', 'V307', 'V309', 'V310', 'V311', 'V312', 'V306', 'V308', 'C3', 'C4', 'V319', 'V303', 'V304', 'C5', 'V320', 'C13', 'C12', 'C11', 'V321', 'C8', 'C9', 'C10', 'C7', 'C6', 'V280', 'V285', 'V287', 'V279', 'V284', 'C14', 'V291', 'V290', 'V286', 'V292', 'V295', 'V297', 'V294', 'V293', 'V299', 'V298', 'V302', 'ProductCD', 'TransactionAmt', 'TransactionDT', 'isFraud', 'TransactionID']
0.05: ['V102', 'V100', 'V101', 'V98', 'V97', 'V96', 'V95', 'V123', 'V124', 'V121', 'V122', 'V120', 'V119', 'V117', 'V118', 'V110', 'V109', 'V116', 'V99', 'V131', 'V128', 'V129', 'V127', 'V134', 'V133', 'V132', 'V126', 'V135', 'V125', 'V136', 'V137', 'V130', 'V103', 'V104', 'V115', 'V114', 'V113', 'V112', 'V111', 'V108', 'V106', 'V105', 'V107']
0.21: ['V313', 'V314', 'V315', 'V300', 'V301', 'V282', 'V281', 'V283', 'V289', 'V288', 'V296', 'D1']
0.27: ['card3', 'card6', 'card4']
0.72: ['card5']
1.51: ['card2']
11.13: ['addr1', 'addr2']
12.87: ['D10']
12.88: ['V

In [33]:
print("total null cols:")
for pct, cols in null_pct_cols.items():
    print(f"{pct}: {len(cols)}")


total null cols:
0.0: 52
0.05: 43
0.21: 12
0.27: 3
0.72: 1
1.51: 1
11.13: 2
12.87: 1
12.88: 23
13.06: 22
15.09: 1
15.1: 20
15.99: 1
28.6: 1
28.61: 18
28.68: 1
44.51: 1
45.91: 3
47.29: 12
47.55: 1
47.66: 1
52.47: 1
58.63: 2
58.64: 1
59.35: 1
59.65: 1
76.05: 16
76.32: 19
76.36: 31
76.75: 1
77.91: 46
86.05: 18
86.12: 29
87.31: 2
87.61: 1
89.04: 1
89.47: 1
89.51: 1
93.41: 1
93.63: 1


In [34]:
print("v subgroup count in each null group:")
v_cols_tot = 0
v_sub = {}
for pct, cols in null_pct_cols.items():
    v_cols = (len([col for col in cols if col.startswith('V')]))
    v_cols_tot += v_cols
    v_sub[pct] = v_cols
print(f"total v cols: {v_cols_tot}") # for checking
v_subgroup_sorted = dict(sorted(v_sub.items(), key=lambda x: x[1], reverse=True))
v_subgroup_sorted = pd.DataFrame.from_dict(v_subgroup_sorted, orient='index', columns=['v_counts'])
print(v_subgroup_sorted)

v subgroup count in each null group:
total v cols: 339
       v_counts
77.91        46
0.05         43
0.00         32
76.36        31
86.12        29
12.88        23
13.06        22
15.10        20
76.32        19
28.61        18
86.05        18
76.05        16
0.21         11
47.29        11
0.27          0
0.72          0
1.51          0
11.13         0
12.87         0
15.09         0
15.99         0
28.60         0
28.68         0
44.51         0
45.91         0
47.55         0
47.66         0
52.47         0
58.63         0
58.64         0
59.35         0
59.65         0
76.75         0
87.31         0
87.61         0
89.04         0
89.47         0
89.51         0
93.41         0
93.63         0


kernel resets every time i close pycharm so I'd have to rerun the whole pipeline again. pickling train after downcast+merge means i can just load it back instantly next time instead of redoing everything

In [35]:
import os
os.makedirs('../data/processed', exist_ok=True)

In [36]:
train.to_pickle('../data/processed/train_optimized.pkl')

In [37]:
import pandas as pd
import numpy as np

In [38]:

train = pd.read_pickle('../data/processed/train_optimized.pkl')
print(train.shape)
print(f'{train.memory_usage(deep=True).sum():,}')

(590540, 434)
953,877,651


thought more nulls = more fraud, turned out backwards. median is 211 for non-fraud vs 139 for fraud. fraudsters fill out more fields to look legit, normal users just never had identity data collected most of the time

In [39]:

train['null_count_per_row'] = train.isnull().sum(axis=1)

print(train.groupby('isFraud')['null_count_per_row'].mean())

isFraud
0    196.852849
1    161.697817
Name: null_count_per_row, dtype: float64


C:\Users\moizs\AppData\Local\Temp\ipykernel_1816\3449950776.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train['null_count_per_row'] = train.isnull().sum(axis=1)


In [40]:
print(train.groupby('isFraud')['null_count_per_row'].median())

isFraud
0    211.0
1    139.0
Name: null_count_per_row, dtype: float64


fraud rate is 3.51% below $1000 vs 2.46% above - not a huge gap, basically close to the overall 3.5% average. most fraud happens below $1000 just because most transactions happen below $1000 in general, not because small amounts are riskier

In [41]:
print(train[train['TransactionAmt'] > 5000][['TransactionAmt', 'isFraud']].sort_values('TransactionAmt', ascending=False))


        TransactionAmt  isFraud
274339    31937.390625        0
274336    31937.390625        0
296021     6450.970215        0
248413     6085.229980        0
384603     5543.229980        0
275529     5420.000000        0
275535     5420.000000        0
584767     5366.819824        0
315172     5279.950195        0
303106     5279.950195        0
462514     5279.950195        0
171451     5278.950195        0
575569     5277.950195        0
409855     5191.000000        0
422708     5191.000000        1
119566     5094.950195        0
423729     5047.470215        0
584835     5001.819824        0


In [42]:
print(train[train['TransactionAmt'] > 1000]['isFraud'].mean())
print(train[train['TransactionAmt'] <= 1000]['isFraud'].mean())

0.02463189761937526
0.035119060885725896


In [43]:
fraud_above = train[train['TransactionAmt'] > 1000]['isFraud'].sum()
fraud_below = train[train['TransactionAmt'] <= 1000]['isFraud'].sum()
print(f'fraud below 1000: {fraud_below}')
print(f'fraud above 1000: {fraud_above}')

fraud below 1000: 20484
fraud above 1000: 179


In [44]:
total_frauds = fraud_below+fraud_above
print(f'total frauds: {total_frauds}')


total frauds: 20663


In [45]:
pct_below_fraud = ((fraud_below/total_frauds)*100).round(2)
pct_above_fraud = ((fraud_above/total_frauds)*100).round(2)
print(f'Fraud above 1000: {pct_above_fraud}\nFraud below 1000: {pct_below_fraud}')

Fraud above 1000: 0.87
Fraud below 1000: 99.13


In [46]:
trans_below = (train['TransactionAmt'] <= 1000).sum()
trans_above = (train['TransactionAmt'] > 1000).sum()
print(f'transaction below 1000: {trans_below}')
print(f'transaction above 1000: {trans_above}')

transaction below 1000: 583273
transaction above 1000: 7267


In [47]:
print(trans_below+trans_above == train_trans_len)
print(trans_below+trans_above == len(train))

True
True


## Class Distribution
| Class     | Count   | Percentage |
| --------- | ------- | ---------- |
| Legit (0) | 569,877 | ~96.5%     |
| Fraud (1) | 20,663  | ~3.5%      |


In [48]:
legitimate = len(train) - total_frauds
print(f'legitimate transaction: {legitimate}')

legitimate transaction: 569877


In [49]:
c1_low = train['C1'] <= train['C1'].median()

In [50]:
c3_low = train['C3'] <= train['C3'].median()

In [51]:
print(c1_low)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [52]:
amt_low = train['TransactionAmt'] <= 1000

In [53]:
print(amt_low.head(20))

0     True
1     True
2     True
3     True
4     True
5     True
6     True
7     True
8     True
9     True
10    True
11    True
12    True
13    True
14    True
15    True
16    True
17    True
18    True
19    True
Name: TransactionAmt, dtype: bool


In [54]:
print(train[c3_low & amt_low]['isFraud'].mean())
print(train[c3_low & ~amt_low]['isFraud'].mean())
print(train[~c3_low & amt_low]['isFraud'].mean())
print(train[~c3_low & ~amt_low]['isFraud'].mean())

0.035257193817294086
0.024638678596008257
0.0020601565718994645
0.0
